# Furiosa LLM 4모델 비교

개발 환자 356명, 같은 5-fold와 임상·Holter 입력 및 train fold 사례를 사용합니다. 검증용 탐색 연구이며 기존 test 89명은 이번 비교에 사용하지 않습니다. 결과는 임상적으로 검증되지 않았습니다. 이 노트북은 결과만 읽고 모델을 다시 실행하지 않습니다.


In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path("/root/.cache/std3_holter/llm_compare")
runs = {
    "Qwen3-32B": "run_20261102_cv5",
    "GPT-OSS-20B": "run_20261103_gpt_oss",
    "Qwen3-30B-A3B": "run_20261103_qwen_moe",
    "EXAONE-4.0-32B": "run_20261103_exaone",
}
rows = []
for model, name in runs.items():
    path = ROOT / name
    if not (path / "metadata.json").is_file():
        print(model, ": 대기 중")
        continue
    meta = json.loads((path / "metadata.json").read_text())
    print(model, ":", meta["status"])
    if (path / "predictions.csv").exists():
        frame = pd.read_csv(path / "predictions.csv")
        display(frame.groupby(["condition", "status"]).size().rename("건수").reset_index())
    if (path / "metrics.json").exists():
        result = json.loads((path / "metrics.json").read_text())
        for condition, values in result.items():
            rows.append({"model": model, "condition": condition, **values["all_successful"]})
# Qwen3-32B 임상 단독은 기존 별도 실험에서 가져옵니다.
p = ROOT / "run_20261102_diagnostics" / "metrics.json"
if p.exists():
    result = json.loads(p.read_text())
    rows.append(
        {
            "model": "Qwen3-32B",
            "condition": "clinical_only",
            **result["clinical_only"]["all_successful"],
        }
    )
if rows:
    display(pd.DataFrame(rows))